3, 16). I denna uppgift ska du göra ett komplett ML-flöde där du modellerar diamantpriser som finns tillgängliga i datasetet “diamonds.csv”. 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import root_mean_squared_error

In [49]:
df = pd.read_csv("diamonds.csv")
df.head()
# Info om dataset
# "price" price in US dollars, "carat" weight of the diamond, "cut" quality of the cut, "color" diamond colour, from J (worst) to D (best)
# "clarity" a measurement of how clear the diamond is (I1 (worst), SI2, SI1, VS2, VS1, VVS2, VVS1, IF (best))
# "x" length in mm, "y" width in mm, "z" depth in mm
# "depth" total depth percentage = z / mean(x, y) = 2 * z / (x + y) (43--79)
# "table" width of top of diamond relative to widest point (43--95)

,Unnamed: 0,carat,cut,color,clarity,depth,table,price,x,y,z
0,1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [50]:
# Det finns inga NULL värden, men det finns 3 kolumner med kategoriska variabler som kräver att vi gör om dem till numeriska variabler. 
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  53940 non-null  int64  
 1   carat       53940 non-null  float64
 2   cut         53940 non-null  object 
 3   color       53940 non-null  object 
 4   clarity     53940 non-null  object 
 5   depth       53940 non-null  float64
 6   table       53940 non-null  float64
 7   price       53940 non-null  int64  
 8   x           53940 non-null  float64
 9   y           53940 non-null  float64
 10  z           53940 non-null  float64
dtypes: float64(6), int64(2), object(3)
memory usage: 4.5+ MB


In [ ]:
# Jag börjar med att ta bort Unnamed kolumnen som inte innehåller någon information som är relevant.
df = df.drop(columns="Unnamed: 0")

# Här delas upp datan i X och y
X = df.drop(columns="price")
y = df["price"]

# Här visas alla unika värden i cut kolumnen, och det finns 5 olika kategorier av cut.
X["cut"].value_counts()

cut
Ideal        21551
Premium      13791
Very Good    12082
Good          4906
Fair          1610
Name: count, dtype: int64

In [52]:
# Samma med color
X["color"].value_counts()

color
G    11292
E     9797
F     9542
H     8304
D     6775
I     5422
J     2808
Name: count, dtype: int64

In [53]:
X["clarity"].value_counts()

clarity
SI1     13065
VS2     12258
SI2      9194
VS1      8171
VVS2     5066
VVS1     3655
IF       1790
I1        741
Name: count, dtype: int64

In [54]:
# Här görs kategoriska variabler om till numeriska variabler med hjälp av get_dummies.
X = pd.get_dummies(X, columns=["cut", "color", "clarity"], drop_first=True)
X.head()

,carat,depth,table,x,y,z,cut_Good,cut_Ideal,cut_Premium,cut_Very Good,...,color_H,color_I,color_J,clarity_IF,clarity_SI1,clarity_SI2,clarity_VS1,clarity_VS2,clarity_VVS1,clarity_VVS2
0,0.23,61.5,55.0,3.95,3.98,2.43,False,True,False,False,...,False,False,False,False,False,True,False,False,False,False
1,0.21,59.8,61.0,3.89,3.84,2.31,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
2,0.23,56.9,65.0,4.05,4.07,2.31,True,False,False,False,...,False,False,False,False,False,False,True,False,False,False
3,0.29,62.4,58.0,4.20,4.23,2.63,False,False,True,False,...,False,True,False,False,False,False,False,True,False,False
4,0.31,63.3,58.0,4.34,4.35,2.75,True,False,False,False,...,False,False,True,False,False,True,False,False,False,False


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LinearRegression()

# Här görs en cross validation på träningsdatan för att få en bättre uppfattning om hur modellen presterar.
# Jag använder 5-fold cross validation. Cross validation gör att modellen testas flera gånger på olika delar av datan.
cv_scores = cross_val_score(lr, X_train, y_train, cv=5, scoring='neg_mean_squared_error')

# cross_val_score returnerar negativa MSE-värden, så jag tar minus och kvadratroten av den för att få RMSE. 
rmse_cv_scores = np.sqrt(-cv_scores)
print("RMSE per iteration:", rmse_cv_scores)
print(f"Average RMSE av cross validation: {rmse_cv_scores.mean():.2f}")

RMSE per fold: [1090.63946473 1153.83549746 1170.40650064 1114.0357523  1121.17205526]
Average RMSE av cross validation: 1130.02


In [ ]:
# Efter att ha utvärderat modellen med cross validation, tränar jag den på hela träningsdatan och predikterar på testdatan.
lr.fit(X_train, y_train)
pred = lr.predict(X_test)

# Modellen presterar ganska lika på testdatan som i cross validation, vilket är ett bra tecken på att modellen inte överanpassar.
rmse_test = root_mean_squared_error(y_test, pred)
print(f"RMSE for test data: {rmse_test:.2f}")

RMSE for test data: 1135.21
